# AEGEAN | Trustpilot - Apify Scraper

Retrieve Apify reviews and raw backups, merge records and report historical coverage.

**Dependencies:** apify-client pandas. Install in the active kernel before running.

**Execution:** Run the setup, configuration and collection cells in order. Review airline selections and limits before collection. Existing data must be migrated to `Data/Scraping/trustpilot_reviews/` before resuming.

**Outputs:** All generated files are stored in `Data/Scraping/trustpilot_reviews/`. Existing filenames and schemas are retained.

**Cost warning:** Collection may start paid Apify jobs. Configure `APIFY_TOKEN` or enter it securely when prompted.

## 1. Portable project paths

In [ ]:
from pathlib import Path

# Locate the project from the notebook kernel's working directory.
# Start Jupyter inside the project tree; the root name is not significant.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if all((p / name).is_dir() for name in ("Python Code", "Data", "Streamlit"))),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the project folder containing Python Code, Data and Streamlit.")

SCRAPING_DIR = PROJECT_ROOT / "Data" / "Scraping"
SOURCE_DIR = SCRAPING_DIR / 'trustpilot_reviews'
SOURCE_DIR.mkdir(parents=True, exist_ok=True)


## Imports and prerequisites

In [ ]:
# ================================================================
# TRUSTPILOT SCRAPER — MULTIPLE AIRLINES
# MAX_REVIEWS = 1000
#
# Airlines:
#   - Aegean Airlines
#   - Air France
#   - Lufthansa
#   - Turkish Airlines
#
# Features:
#   1. Select which airlines to scrape
#   2. MAX_REVIEWS = 1000
#   3. Reviews from 2024-01-01 onwards
#   4. All languages
#   5. Individual CSV files for each airline
#   6. Raw JSON backup for every Apify run
#   7. Incremental merging and deduplication
#   8. Per-airline historical coverage reports
#   9. Combined Trustpilot dataset
#  10. Recovery of existing Apify runs
#  11. Continue processing if one airline fails
# ================================================================

import os
import json
import getpass
import traceback

from pathlib import Path
from datetime import datetime, timezone, timedelta

import pandas as pd
from apify_client import ApifyClient
from IPython.display import display

## 1. Configuration

In [ ]:
ACTOR_ID = "rastriq/trustpilot-reviews-scraper"

AIRLINE_URLS = {

    "Aegean Airlines":
        "https://www.trustpilot.com/review/aegeanair.com",

    "Air France":
        "https://www.trustpilot.com/review/www.airfrance.com",

    "Lufthansa":
        "https://www.trustpilot.com/review/www.lufthansa.com",

    "Turkish Airlines":
        "https://www.trustpilot.com/review/www.turkishairlines.com",
    
    "Bangkok Airways":
        "https://www.trustpilot.com/review/www.bangkokair.com",
    
    "Porter":
        "https://www.trustpilot.com/review/flyporter.com",
    
    "Virgin":
        "https://www.trustpilot.com/review/www.virginaustralia.com"
}


# ------------------------------------------------
# SELECT AIRLINES TO SCRAPE
#
# Comment out any airline you want to exclude.
# Existing data for excluded airlines is preserved.
# ------------------------------------------------

AIRLINES_TO_SCRAPE = [

#    "Aegean Airlines",

#    "Air France",

#    "Lufthansa",

#    "Turkish Airlines",
    
    "Bangkok Airways",
    
    "Porter",
    
    "Virgin"
]


# ------------------------------------------------
# SCRAPING PARAMETERS
# ------------------------------------------------

START_DATE = "2024-01-01"

MAX_REVIEWS = 1000

LANGUAGES = ["all"]

SCRAPE_MODE = "deep"

RUN_TIMEOUT_SECONDS = 7200
CLIENT_TIMEOUT_SECONDS = 7500


# ------------------------------------------------
# RECOVER EXISTING RUNS
#
# If an Apify run finishes but your notebook fails,
# insert the run ID below to retrieve its dataset
# without starting another paid extraction.
#
# Example:
# "Lufthansa": "YOUR_RUN_ID"
# ------------------------------------------------

RECOVER_RUN_IDS = {

    # "Aegean Airlines": "RUN_ID",
    # "Air France": "RUN_ID",
    # "Lufthansa": "RUN_ID",
    # "Turkish Airlines": "RUN_ID"

}


# ------------------------------------------------
# OUTPUT PATHS
# ------------------------------------------------

OUTPUT_DIR = SOURCE_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_FILE = (
    OUTPUT_DIR / "trustpilot_all_airlines.csv"
)

RUN_HISTORY_FILE = (
    OUTPUT_DIR / "trustpilot_run_history.csv"
)

OVERALL_COVERAGE_FILE = (
    OUTPUT_DIR / "trustpilot_coverage_summary.csv"
)

RUN_SUMMARY_FILE = (
    OUTPUT_DIR / "trustpilot_latest_run_summary.csv"
)

## 2. Validate Configuration

In [ ]:
unknown_airlines = set(
    AIRLINES_TO_SCRAPE
) - set(AIRLINE_URLS)

if unknown_airlines:
    raise ValueError(
        f"Unknown airlines: {unknown_airlines}"
    )

if len(AIRLINES_TO_SCRAPE) != len(set(AIRLINES_TO_SCRAPE)):
    raise ValueError(
        "AIRLINES_TO_SCRAPE contains duplicate airlines."
    )

if not AIRLINES_TO_SCRAPE:
    raise ValueError(
        "Select at least one airline to scrape."
    )

print("=" * 75)
print("TRUSTPILOT MULTI-AIRLINE SCRAPER")
print("=" * 75)

print("Selected airlines:")

for airline in AIRLINES_TO_SCRAPE:
    print(" -", airline)

print("\nMAX_REVIEWS:", MAX_REVIEWS)
print("START_DATE:", START_DATE)
print("LANGUAGES:", LANGUAGES)
print("MODE:", SCRAPE_MODE)

## 3. Initialize Apify Client

In [ ]:
APIFY_TOKEN = os.getenv("APIFY_TOKEN")

if not APIFY_TOKEN:
    APIFY_TOKEN = getpass.getpass(
        "Enter Apify API token: "
    )

client = ApifyClient(
    token=APIFY_TOKEN,
    timeout_long=timedelta(
        seconds=CLIENT_TIMEOUT_SECONDS
    ),
    timeout_max=timedelta(
        seconds=CLIENT_TIMEOUT_SECONDS
    )
)

## 4. Helper Functions

In [ ]:
def airline_slug(airline):

    """Convert an airline name into a filename slug."""
    return (
        airline.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


def parse_dates(series):

    """Parse source dates into pandas datetime values."""
    return pd.to_datetime(
        series,
        errors="coerce",
        format="mixed",
        utc=True
    ).dt.tz_localize(None)


def first_available(df, candidates):

    """Select the first available source field from candidate columns."""
    result = pd.Series(
        pd.NA,
        index=df.index,
        dtype="object"
    )

    for column in candidates:

        if column in df.columns:

            result = result.combine_first(
                df[column]
            )

    return result


def normalize_review_ids(series):

    """Normalize review identifiers for consistent matching."""
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def read_existing_csv(path):

    """Load existing CSV records for incremental processing."""
    if not path.exists():
        return pd.DataFrame()

    df = pd.read_csv(
        path,
        dtype={"source_review_id": "string"},
        low_memory=False
    )

    if "review_date" in df.columns:
        df["review_date"] = parse_dates(
            df["review_date"]
        )

    return df


def get_run_field(run, dict_key, object_attribute):

    """Read a run field from the supported response representations."""
    if isinstance(run, dict):
        return run.get(dict_key)

    return getattr(
        run,
        object_attribute,
        None
    )


def merge_reviews(existing, new):

    """Merge old and new review records using the existing deduplication rules."""
    if existing.empty and new.empty:
        return pd.DataFrame()

    df = pd.concat(
        [existing, new],
        ignore_index=True
    )

    df["source_review_id"] = normalize_review_ids(
        df["source_review_id"]
    )

    df["review_date"] = parse_dates(
        df["review_date"]
    )

    df = df[
        df["source_review_id"].notna()
        & df["source_review_id"].ne("")
        & df["review_date"].notna()
    ].copy()

    df = df.drop_duplicates(
        subset=[
            "airline",
            "source",
            "source_review_id"
        ],
        keep="first"
    )

    df = df.sort_values(
        "review_date",
        ascending=False
    ).reset_index(drop=True)

    return df


def get_airline_paths(airline):

    """Construct the output paths for one airline."""
    slug = airline_slug(airline)

    return {

        "csv":
            OUTPUT_DIR / f"{slug}_reviews.csv",

        "coverage":
            OUTPUT_DIR / f"{slug}_coverage_summary.csv",

        "monthly":
            OUTPUT_DIR / f"{slug}_monthly_reviews.csv"
    }

## 5. Standardize Trustpilot Records

In [ ]:
def standardize_reviews(items, airline, run_id):

    """Transform source records into the existing normalized output schema."""
    if not items:
        return pd.DataFrame()

    raw = pd.json_normalize(
        items,
        sep="."
    )

    df = pd.DataFrame(index=raw.index)

    field_map = {

        "source_review_id": [
            "review_id",
            "reviewId",
            "id"
        ],

        "review_date": [
            "published_at",
            "publishedDate",
            "date"
        ],

        "rating": [
            "rating",
            "stars"
        ],

        "review_title": [
            "title",
            "reviewTitle"
        ],

        "review": [
            "text",
            "reviewText",
            "content"
        ],

        "review_url": [
            "source_url",
            "reviewUrl",
            "url"
        ],

        "language": [
            "lang",
            "language"
        ],

        "author": [
            "author_name",
            "authorName"
        ],

        "reviewer_country": [
            "country"
        ],

        "experience_date": [
            "ext_experienced_at"
        ],

        "verified": [
            "ext_review_verified"
        ],

        "company_reply": [
            "developer_reply"
        ],

        "company_reply_date": [
            "developer_reply_at"
        ],

        "helpful_count": [
            "helpful_count"
        ],

        "trust_score": [
            "ext_business_trust_score"
        ],

        "business_review_count": [
            "ext_business_review_count"
        ],

        "business_review_count_12m": [
            "ext_business_review_count_12m"
        ],

        "review_source": [
            "ext_review_source"
        ],

        "review_verification_level": [
            "ext_review_verification_level"
        ],

        "business_website": [
            "ext_business_website"
        ],

        "business_country": [
            "ext_business_country"
        ],

        "business_is_claimed": [
            "ext_business_is_claimed"
        ]
    }

    for target, candidates in field_map.items():

        df[target] = first_available(
            raw,
            candidates
        )

    df["source_review_id"] = normalize_review_ids(
        df["source_review_id"]
    )

    df["review_date"] = parse_dates(
        df["review_date"]
    )

    df["experience_date"] = parse_dates(
        df["experience_date"]
    )

    df["rating"] = pd.to_numeric(
        df["rating"],
        errors="coerce"
    )

    df["airline"] = airline
    df["source"] = "Trustpilot"
    df["apify_run_id"] = run_id

    df["scraped_at"] = datetime.now(
        timezone.utc
    ).isoformat()

    print("Raw records:", len(raw))

    print(
        "Records with valid IDs:",
        df["source_review_id"].notna().sum()
    )

    print(
        "Records with valid dates:",
        df["review_date"].notna().sum()
    )

    df = df[
        df["source_review_id"].notna()
        & df["source_review_id"].ne("")
        & df["review_date"].notna()
    ].copy()

    df = df[
        df["review_date"]
        >= pd.Timestamp(START_DATE)
    ].copy()

    df = df.drop_duplicates(
        subset=[
            "airline",
            "source_review_id"
        ],
        keep="first"
    )

    print(
        "Valid unique reviews:",
        len(df)
    )

    return df.reset_index(drop=True)

## 6. Run Actor Or Recover Existing Run

In [ ]:
def run_apify_actor(airline):

    """Run or recover the configured Apify extraction."""
    url = AIRLINE_URLS[airline]

    print("\n" + "=" * 75)
    print("AIRLINE:", airline)
    print("URL:", url)
    print("MAX_REVIEWS:", MAX_REVIEWS)
    print("=" * 75)

    existing_run_id = RECOVER_RUN_IDS.get(
        airline
    )

    if existing_run_id:

        print(
            "\nRecovering existing run:",
            existing_run_id
        )

        run = client.run(
            existing_run_id
        ).get()

    else:

        run_input = {

            "companies": [url],

            "mode": SCRAPE_MODE,

            "maxReviews": MAX_REVIEWS,

            "languages": LANGUAGES,

            "dateFrom": START_DATE,

            "hideAuthorNames": False
        }

        print("\nActor input:")
        print(
            json.dumps(
                run_input,
                indent=2
            )
        )

        print("\nStarting Apify actor...")

        run = client.actor(
            ACTOR_ID
        ).call(
            run_input=run_input,
            run_timeout=timedelta(
                seconds=RUN_TIMEOUT_SECONDS
            ),
            timeout=timedelta(
                seconds=CLIENT_TIMEOUT_SECONDS
            )
        )

    if run is None:
        raise RuntimeError(
            "Apify returned no run information."
        )

    run_id = get_run_field(
        run,
        "id",
        "id"
    )

    status = get_run_field(
        run,
        "status",
        "status"
    )

    dataset_id = get_run_field(
        run,
        "defaultDatasetId",
        "default_dataset_id"
    )

    print("\nRun status:", status)
    print("Run ID:", run_id)

    if not dataset_id:
        raise ValueError(
            "Apify run has no dataset ID."
        )

    items = list(
        client.dataset(
            dataset_id
        ).iterate_items()
    )

    print(
        "Dataset records retrieved:",
        len(items)
    )

    raw_file = OUTPUT_DIR / (
        f"{airline_slug(airline)}_{run_id}_raw.json"
    )

    with open(
        raw_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            items,
            f,
            ensure_ascii=False,
            indent=2,
            default=str
        )

    print(
        "Raw JSON saved:",
        raw_file.resolve()
    )

    return run_id, status, items

## 7. Historical Coverage

In [ ]:
def create_coverage_report(df, airline):

    """Calculate and persist airline review coverage statistics."""
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()

    paths = get_airline_paths(airline)

    dates = parse_dates(
        df["review_date"]
    )

    yearly = dates.dt.year.value_counts()

    summary = pd.DataFrame([{

        "airline": airline,

        "total_reviews": len(df),

        "earliest_date": dates.min(),

        "latest_date": dates.max(),

        "reviews_2024": int(
            yearly.get(2024, 0)
        ),

        "reviews_2025": int(
            yearly.get(2025, 0)
        ),

        "reviews_2026": int(
            yearly.get(2026, 0)
        ),

        "missing_text": int(
            df["review"]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        ),

        "missing_rating": int(
            df["rating"].isna().sum()
        )
    }])

    monthly = (
        df.groupby(
            dates.dt.to_period("M")
        )
        .size()
        .rename("reviews")
    )

    all_months = pd.period_range(
        pd.Timestamp(START_DATE).to_period("M"),
        pd.Timestamp.now().to_period("M"),
        freq="M"
    )

    monthly = monthly.reindex(
        all_months,
        fill_value=0
    ).reset_index()

    monthly.columns = [
        "month",
        "reviews"
    ]

    monthly["month"] = (
        monthly["month"].astype(str)
    )

    summary.to_csv(
        paths["coverage"],
        index=False,
        encoding="utf-8-sig"
    )

    monthly.to_csv(
        paths["monthly"],
        index=False,
        encoding="utf-8-sig"
    )

    return summary, monthly

## 8. Process One Airline

In [ ]:
def process_airline(airline):

    """Process one airline and persist its incremental results."""
    paths = get_airline_paths(airline)

    previous = read_existing_csv(
        paths["csv"]
    )

    previous_ids = set()

    if not previous.empty:

        previous_ids = set(
            normalize_review_ids(
                previous["source_review_id"]
            ).dropna()
        )

    print(
        "\nPreviously saved reviews:",
        len(previous)
    )

    run_id, status, items = run_apify_actor(
        airline
    )

    if not items:
        raise ValueError(
            "No records returned by actor."
        )

    new_reviews = standardize_reviews(
        items,
        airline,
        run_id
    )

    if new_reviews.empty:
        raise ValueError(
            "No valid reviews returned."
        )

    new_ids = set(
        normalize_review_ids(
            new_reviews["source_review_id"]
        ).dropna()
    )

    overlapping_ids = new_ids & previous_ids
    added_ids = new_ids - previous_ids

    print("\nNEW EXTRACTION ANALYSIS")

    print(
        "Reviews in current run:",
        len(new_reviews)
    )

    print(
        "Already collected:",
        len(overlapping_ids)
    )

    print(
        "Genuinely new review IDs:",
        len(added_ids)
    )

    # Merge previous and new reviews
    df_airline = merge_reviews(
        previous,
        new_reviews
    )

    df_airline.to_csv(
        paths["csv"],
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Total unique reviews after merge:",
        len(df_airline)
    )

    # Historical coverage
    summary, monthly = create_coverage_report(
        df_airline,
        airline
    )

    print("\nCOVERAGE SUMMARY")
    display(summary)

    print("\nMONTHLY REVIEW COUNTS")
    display(monthly)

    return {

        "airline": airline,

        "run_id": run_id,

        "status": str(status),

        "max_reviews": MAX_REVIEWS,

        "reviews_in_run": len(new_reviews),

        "previous_reviews": len(previous),

        "overlapping_reviews": len(overlapping_ids),

        "new_unique_reviews": len(added_ids),

        "total_unique_reviews": len(df_airline),

        "earliest_date": df_airline[
            "review_date"
        ].min(),

        "latest_date": df_airline[
            "review_date"
        ].max(),

        "reviews_2024": int(
            (
                df_airline["review_date"].dt.year
                == 2024
            ).sum()
        ),

        "reviews_2025": int(
            (
                df_airline["review_date"].dt.year
                == 2025
            ).sum()
        ),

        "reviews_2026": int(
            (
                df_airline["review_date"].dt.year
                == 2026
            ).sum()
        )
    }

## 9. Update Combined Trustpilot Dataset

In [ ]:
def update_combined_dataset():

    """Combine saved airline datasets into the master export."""
    frames = []

    for airline in AIRLINE_URLS:

        paths = get_airline_paths(airline)

        if paths["csv"].exists():

            df = read_existing_csv(
                paths["csv"]
            )

            if not df.empty:
                frames.append(df)

    if not frames:
        return pd.DataFrame()

    combined = merge_reviews(
        pd.DataFrame(),
        pd.concat(
            frames,
            ignore_index=True
        )
    )

    combined.to_csv(
        COMBINED_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n" + "=" * 75)
    print("COMBINED TRUSTPILOT DATASET")
    print("=" * 75)

    print(
        "Total unique reviews:",
        len(combined)
    )

    print("\nReviews by airline:")
    print(
        combined["airline"].value_counts()
    )

    print(
        "\nSaved:",
        COMBINED_FILE.resolve()
    )

    return combined

## 10. Main Execution

In [ ]:
def main():

    """Orchestrate collection and report the resulting dataset."""
    run_results = []

    print("\n" + "=" * 75)
    print("STARTING MULTI-AIRLINE EXTRACTION")
    print("=" * 75)

    for index, airline in enumerate(
        AIRLINES_TO_SCRAPE,
        start=1
    ):

        print("\n" + "#" * 75)

        print(
            f"PROCESSING AIRLINE "
            f"{index}/{len(AIRLINES_TO_SCRAPE)}"
        )

        print("AIRLINE:", airline)

        print("#" * 75)

        try:

            result = process_airline(
                airline
            )

            run_results.append(
                result
            )

            # Save progress after each successful airline
            pd.DataFrame(
                run_results
            ).to_csv(
                RUN_SUMMARY_FILE,
                index=False,
                encoding="utf-8-sig"
            )

            print(
                f"\nSUCCESS: {airline}"
            )

        except Exception as e:

            print(
                f"\nERROR processing {airline}: {e}"
            )

            traceback.print_exc()

            run_results.append({

                "airline": airline,

                "status": "ERROR",

                "error": str(e)
            })

            pd.DataFrame(
                run_results
            ).to_csv(
                RUN_SUMMARY_FILE,
                index=False,
                encoding="utf-8-sig"
            )

            print(
                "\nContinuing with next airline..."
            )

    # Update combined dataset using ALL available
    # airline CSVs, including excluded airlines.
    df_trustpilot_all = update_combined_dataset()

    results_df = pd.DataFrame(
        run_results
    )

    print("\n" + "=" * 75)
    print("FINAL RUN SUMMARY")
    print("=" * 75)

    display(results_df)

    print("\nSaved run summary:")
    print(RUN_SUMMARY_FILE.resolve())

    return results_df, df_trustpilot_all

## 11. Execute

In [ ]:
results_df, df_trustpilot_all = main()